# Synthetic Q&A from a documentThis notebook generates a fine-tuning dataset by asking a teacher model to write Q&A pairs from a source document, runs an LLM-judge quality filter, then fine-tunes a smaller student model on the result.**You'll end up with**: a leaderboard showing whether your source document + chosen models can produce a SHIP-worthy fine-tune for Q&A, plus the trained model(s) themselves.**Time**: ~30–60 minutes.**Cost**: ~$5–15 per run depending on `NUM_EXAMPLES` and source size.---## What you need1. **A source document** — PDF, markdown, or plain text. This notebook ships with a small COBOL sample; replace `SOURCE_DOC` with your own.2. Environment variables:   - `AZURE_AI_PROJECT_ENDPOINT`, `OPENAI_BASE_URL`, `AZURE_OPENAI_API_KEY`   - (Optional) `AZURE_CONTENT_SAFETY_ENDPOINT` + `AZURE_CONTENT_SAFETY_KEY` for the Phase 2b pre-screen3. `FINETUNING_SKILL_PATH` — path to the skill's `Skills/` folder4. A teacher model deployment (e.g. `gpt-4.1`, `gpt-5.4`) — used both for Q&A generation and the quality-filter judge5. A student model deployment that supports fine-tuning (e.g. `gpt-4.1-mini`)

In [ ]:
import os, json, subprocess, sysfrom pathlib import PathPROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]BASE_URL         = os.environ["OPENAI_BASE_URL"]API_KEY          = os.environ["AZURE_OPENAI_API_KEY"]SKILL            = Path(os.environ.get("FINETUNING_SKILL_PATH", "../../../Skills"))WORK             = Path("./run").resolve()WORK.mkdir(exist_ok=True)SOURCE_DOC    = Path("fixtures/sample_source.md")  # REPLACE with your own PDF or .mdTEACHER_MODEL = "gpt-4.1"     # used for both Q&A generation and the quality-filter judgeSTUDENT_MODEL = "gpt-4.1-mini"TASK_NAME     = "doc-qna"NUM_EXAMPLES  = 200            # bump to 2000+ for production-quality runsprint(f"Source:   {SOURCE_DOC} ({SOURCE_DOC.stat().st_size:,} bytes)")print(f"Teacher:  {TEACHER_MODEL}")print(f"Student:  {STUDENT_MODEL}")print(f"Examples: {NUM_EXAMPLES}")print(f"Work dir: {WORK}")

## 1. Extract text from PDF (if needed) and upload to FoundryIf your source is already markdown or text, skip the extraction. PDF inputs get extracted with `pypdf`. The Foundry file API caps `user_data` uploads at 20MB, so very large PDFs may need chunking — for most reference docs the text is well under that even when the PDF is huge.

In [ ]:
from openai import OpenAIclient = OpenAI(base_url=BASE_URL, api_key=API_KEY)# Extract text if PDFif SOURCE_DOC.suffix.lower() == ".pdf":    from pypdf import PdfReader    text_path = WORK / (SOURCE_DOC.stem + ".txt")    reader = PdfReader(SOURCE_DOC)    pages = [p.extract_text() or "" for p in reader.pages]    text_path.write_text("\n\n".join(pages), encoding="utf-8")    print(f"Extracted {len(pages)} pages → {text_path} ({text_path.stat().st_size:,} bytes)")else:    text_path = SOURCE_DOC    print(f"Using source as-is: {text_path}")with open(text_path, "rb") as fh:    src_file = client.files.create(file=(text_path.name, fh), purpose="user_data")print(f"\nUploaded: {src_file.id}")

## 2. Run the autopilot end-to-endThis shells out to `auto_finetune.py auto` with:- `--datagen-backend foundry-file` + `--datagen-file-id` — generate Q&A from our uploaded doc- `--content-safety-prescreen` (Phase 2b) — drop rows that trip Azure FT's safety check (rare but happens)- `--quality-filter` (Phase 2c) — LLM-judge each row on `non_fragmented` / `non_empty` / `on_topic`, drop below threshold 4- Three default candidates spanning epochs/lr/student-modelYou'll see phase-by-phase progress. The autopilot writes `review_iter1.json` with the SHIP/ITERATE/STOP decision and per-candidate diagnostics.

In [ ]:
# Skip content-safety prescreen if you don't have those env varsUSE_CS_PRESCREEN = bool(os.environ.get("AZURE_CONTENT_SAFETY_ENDPOINT") and os.environ.get("AZURE_CONTENT_SAFETY_KEY"))cmd = [    sys.executable, str(SKILL / "scripts" / "auto_finetune.py"), "auto",    "--description", "Q&A model for the source document. Answers technical questions accurately, "                     "citing structure/section terminology where appropriate.",    "--task-name", TASK_NAME,    "--model", STUDENT_MODEL,    "--teacher", TEACHER_MODEL,    "--datagen-backend", "foundry-file",    "--datagen-file-id", src_file.id,    "--num-examples", str(NUM_EXAMPLES),    "--max-iterations", "1",    "--max-budget", "20",    "--work-dir", str(WORK),    "--tier", "globalStandard",    "--project-endpoint", PROJECT_ENDPOINT,    "--base-url", BASE_URL,    "--api-key", API_KEY,    "--quality-filter", "--quality-filter-judge", TEACHER_MODEL, "--quality-filter-threshold", "4",]if USE_CS_PRESCREEN:    cmd += ["--content-safety-prescreen", "--content-safety-threshold", "2"]proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)for line in proc.stdout:    print(line, end="")proc.wait()print(f"\n[exit {proc.returncode}]")

## 3. Inspect the leaderboardThe autopilot writes `review_iter1.json` with the SHIP/ITERATE/STOP decision. If the decision is ITERATE, you'll see specific diagnostic recommendations for the next iteration (different HPs, more data, different base model).

In [ ]:
review = json.loads((WORK / "review_iter1.json").read_text())print(f"DECISION: {review['decision']}")print(f"REASON  : {review.get('reason', '(none)')}")print()print("Per-candidate leaderboard:")print(f"  {'candidate':<25} {'combined':>9} {'pass_rate':>10} {'lift':>10}")print(f"  {'-'*25} {'-'*9} {'-'*10} {'-'*10}")for c in review.get("candidates", []):    lift = c.get("lift_pct")    lift_s = f"{lift:+.1f}%" if lift is not None else "—"    print(f"  {c['candidate']:<25} {c.get('combined', 0):>9.2f} {c.get('pass_rate', 0):>9.1f}% {lift_s:>10}")if review["decision"] == "SHIP":    print(f"\n✅ Shipped: {review.get('winner', {}).get('model_id', '?')}")elif review["decision"] == "ITERATE":    print("\n🔄 No candidate met the lift threshold. Diagnostics:")    for step in review.get("next_steps", []):        print(f"  • {step}")

## When ITERATE happensThe autopilot returns ITERATE when no candidate clears the lift threshold. Common diagnoses and what to try:| Diagnosis | What to try ||-----------|------------|| All candidates regressed | Check (a) labels are sensible, (b) the eval judge matches the training task, (c) try a larger base model || One candidate marginally improved but didn't hit threshold | Narrow HPs around that candidate; train longer; more training data || Overfitting (val ratio > 1.5x) | Lower LR (`--candidate-lr 0.5`), fewer epochs, or deploy an earlier checkpoint || All candidates failed FT (cost: 0) | Check upstream FT service status and your quota |**Q&A specifically**: pure-knowledge tasks ("ask any question about X") are hard to FT for small models because the small model genuinely lacks the knowledge. Tasks where FT shines are *output-style* tasks: enforcing a format, terminology, persona, or structured output.## Cleanup```pythonclient.files.delete(src_file.id)   # free the user_data quota```